# 🏠 Data Engineering & Machine Learning avec Snowflake

## Pipeline ML complet — Prédiction du Prix de l'Immobilier

Ce notebook couvre l'ensemble du pipeline :
1. **Configuration** de l'environnement Snowflake
2. **Ingestion** des données depuis S3
3. **Exploration** des données (EDA)
4. **Préparation** des features (encodage, normalisation, split)
5. **Entraînement** de plusieurs modèles ML
6. **Évaluation** des performances
7. **Optimisation** des hyperparamètres
8. **Stockage** du meilleur modèle dans le Model Registry
9. **Inférence** sur de nouvelles données
10. **Application Streamlit** pour les utilisateurs métier

---
**Dataset :** House Price Prediction — `s3://logbrain-datalake/datasets/house_price/`

---
## Étape 1 — Configuration de l'environnement

On commence par créer la base de données, le schéma, le file format CSV et le stage externe pointant vers le bucket S3.

> **Packages requis :** `scikit-learn`, `xgboost`, `modin`, `snowflake-ml-python`  
> Ajoutez-les via le menu **Packages** en haut à droite du notebook.

In [ ]:
-- Création de la base de données et du schéma
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
USE DATABASE HOUSE_PRICE_DB;

CREATE SCHEMA IF NOT EXISTS ML_SCHEMA;
USE SCHEMA ML_SCHEMA;

-- File format CSV
CREATE FILE FORMAT IF NOT EXISTS CSV_FORMAT
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    RECORD_DELIMITER = '\n'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('');

-- Stage externe vers le bucket S3
CREATE STAGE IF NOT EXISTS HOUSE_PRICE_STAGE
    FILE_FORMAT = CSV_FORMAT
    URL = 's3://logbrain-datalake/datasets/house_price/';

-- Vérification des fichiers disponibles dans le stage
LS @HOUSE_PRICE_STAGE;

In [ ]:
from snowflake.snowpark.context import get_active_session
import warnings
warnings.filterwarnings('ignore')

session = get_active_session()
print(f"✅ Session Snowflake active : {session.get_current_database()}.{session.get_current_schema()}")

---
## Étape 2 — Ingestion et exploration initiale des données

Lecture du CSV depuis le stage S3 avec **Snowpark** et création d'une table persistante.

In [ ]:
-- Créer un file format JSON
CREATE OR REPLACE FILE FORMAT JSON_FORMAT
    TYPE = 'JSON'
    STRIP_OUTER_ARRAY = TRUE;

-- Créer un stage JSON
CREATE OR REPLACE STAGE HOUSE_PRICE_JSON_STAGE
    FILE_FORMAT = JSON_FORMAT
    URL = 's3://logbrain-datalake/datasets/house_price/';

-- Charger le JSON brut dans une table de staging
CREATE OR REPLACE TABLE HOUSE_PRICE_RAW (raw_data VARIANT);

COPY INTO HOUSE_PRICE_RAW
FROM @HOUSE_PRICE_JSON_STAGE;

-- Parser le JSON et créer la table finale
CREATE OR REPLACE TABLE HOUSE_PRICE AS
SELECT
    raw_data:"price"::FLOAT          AS PRICE,
    raw_data:"area"::FLOAT           AS AREA,
    raw_data:"bedrooms"::INT         AS BEDROOMS,
    raw_data:"bathrooms"::INT        AS BATHROOMS,
    raw_data:"stories"::INT          AS STORIES,
    raw_data:"mainroad"::VARCHAR     AS MAINROAD,
    raw_data:"guestroom"::VARCHAR    AS GUESTROOM,
    raw_data:"basement"::VARCHAR     AS BASEMENT,
    raw_data:"hotwaterheating"::VARCHAR AS HOTWATERHEATING,
    raw_data:"airconditioning"::VARCHAR AS AIRCONDITIONING,
    raw_data:"parking"::INT          AS PARKING,
    raw_data:"prefarea"::VARCHAR     AS PREFAREA,
    raw_data:"furnishingstatus"::VARCHAR AS FURNISHINGSTATUS
FROM HOUSE_PRICE_RAW;

SELECT COUNT(*) AS nb_lignes FROM HOUSE_PRICE;

In [ ]:
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import (
    StructType, StructField, FloatType, IntegerType, StringType
)

# Charger la table dans un DataFrame Snowpark
df_sp = session.table("HOUSE_PRICE")

print("📋 Schéma de la table :")
df_sp.printSchema()

print(f"\n📊 Nombre de lignes : {df_sp.count()}")
df_sp.show(5)

In [ ]:
# Statistiques descriptives via Snowpark
print("📈 Statistiques descriptives :")
df_sp.describe().show()

In [ ]:
# Vérification des valeurs manquantes
import pandas as pd

df_pd = df_sp.to_pandas()

missing = df_pd.isnull().sum()
print("🔍 Valeurs manquantes par colonne :")
print(missing[missing > 0] if missing.sum() > 0 else "✅ Aucune valeur manquante")

In [ ]:
-- Aperçu des données
SELECT * FROM HOUSE_PRICE LIMIT 10;

---
## Étape 3 — Exploration des données (EDA)

Comprendre la distribution des variables, la corrélation entre les features et la variable cible `PRICE`.

In [ ]:
import streamlit as st
import altair as alt
import pandas as pd

alt.data_transformers.enable('json')
alt.theme.enable('opaque')

# --- Distribution du prix (variable cible) ---
st.subheader("📊 Distribution de la variable cible : PRICE")

hist_price = alt.Chart(df_pd).mark_bar(color='#1f77b4').encode(
    x=alt.X('PRICE:Q', bin=alt.Bin(maxbins=30), title='Prix (USD)'),
    y=alt.Y('count()', title='Nombre de maisons')
).properties(title='Distribution des prix de vente', width=600, height=300)

st.altair_chart(hist_price, use_container_width=True)

In [ ]:
# --- Corrélation des features numériques avec PRICE ---
st.subheader("📊 Corrélation entre les features numériques et PRICE")

numerical_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']
corr_matrix = df_pd[numerical_cols].corr().reset_index().melt('index')
corr_matrix.columns = ['Variable X', 'Variable Y', 'Corrélation']

heatmap = alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('Variable X:N'),
    y=alt.Y('Variable Y:N'),
    color=alt.Color('Corrélation:Q', scale=alt.Scale(scheme='redblue', domain=[-1, 1])),
    tooltip=['Variable X', 'Variable Y', 'Corrélation']
).properties(title='Matrice de corrélation', width=400, height=350)

st.altair_chart(heatmap, use_container_width=True)

# Affichage numérique de la corrélation avec PRICE
st.write("**Corrélation avec PRICE :**")
st.dataframe(df_pd[numerical_cols].corr()[['PRICE']].sort_values('PRICE', ascending=False).round(3))

In [ ]:
# --- Scatter plot AREA vs PRICE ---
st.subheader("📊 Surface vs Prix")

scatter = alt.Chart(df_pd).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('AREA:Q', title='Surface (m²)'),
    y=alt.Y('PRICE:Q', title='Prix (USD)'),
    color=alt.Color('FURNISHINGSTATUS:N', title='Ameublement'),
    tooltip=['AREA', 'PRICE', 'BEDROOMS', 'BATHROOMS', 'FURNISHINGSTATUS']
).properties(title='Surface vs Prix par statut d\'ameublement', width=600, height=350)

st.altair_chart(scatter, use_container_width=True)

In [ ]:
# --- Prix moyen par variables catégorielles ---
st.subheader("📊 Prix moyen selon les variables catégorielles")

cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
            'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']

for col in cat_cols:
    avg_price = df_pd.groupby(col)['PRICE'].mean().reset_index()
    avg_price.columns = [col, 'PRIX_MOYEN']

    bar = alt.Chart(avg_price).mark_bar().encode(
        x=alt.X(f'{col}:N'),
        y=alt.Y('PRIX_MOYEN:Q', title='Prix moyen (USD)'),
        color=alt.Color(f'{col}:N'),
        tooltip=[col, 'PRIX_MOYEN']
    ).properties(title=f'Prix moyen par {col}', width=300, height=200)

    st.altair_chart(bar, use_container_width=False)

---
## Étape 4 — Préparation des données (Feature Engineering)

- **Encodage** des variables catégorielles binaires (`yes`/`no` → `1`/`0`) et de `FURNISHINGSTATUS` (OrdinalEncoder)
- **Normalisation** des features numériques (StandardScaler)
- **Split** train/test (80/20)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Rechargement propre depuis Snowflake
df = session.table("HOUSE_PRICE").to_pandas()

# ── Encodage des colonnes binaires yes/no ──────────────────────────────
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
               'AIRCONDITIONING', 'PREFAREA']

for col in binary_cols:
    df[col] = df[col].str.strip().str.lower().map({'yes': 1, 'no': 0})

# ── Encodage ordinal de FURNISHINGSTATUS ──────────────────────────────
# furnished > semi-furnished > unfurnished
furnishing_order = [['unfurnished', 'semi-furnished', 'furnished']]
oe = OrdinalEncoder(categories=furnishing_order)
df['FURNISHINGSTATUS'] = oe.fit_transform(
    df[['FURNISHINGSTATUS']].applymap(lambda x: x.strip().lower())
)

# ── Séparation features / cible ───────────────────────────────────────
feature_cols = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'MAINROAD',
                'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING', 'AIRCONDITIONING',
                'PARKING', 'PREFAREA', 'FURNISHINGSTATUS']

X = df[feature_cols]
y = df['PRICE']

# ── Split train / test ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Normalisation ─────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("✅ Préparation des données terminée !")
print(f"   Train : {X_train_scaled.shape[0]} lignes | Test : {X_test_scaled.shape[0]} lignes")
print(f"   Nombre de features : {X_train_scaled.shape[1]}")
print(f"\n   Features : {feature_cols}")

In [ ]:
# Sauvegarde des paramètres du scaler dans Snowflake
scaler_params = pd.DataFrame({
    'FEATURE': feature_cols,
    'MEAN_VAL': scaler.mean_,
    'STD_VAL': scaler.scale_
})
session.create_dataframe(scaler_params).write.mode("overwrite").save_as_table("HOUSE_PRICE_DB.ML_SCHEMA.SCALER_PARAMS")
print("✅ Paramètres du scaler sauvegardés dans SCALER_PARAMS")

In [ ]:
SELECT * FROM HOUSE_PRICE_DB.ML_SCHEMA.SCALER_PARAMS;

In [ ]:
# Sauvegarde des données de test dans Snowflake (pour l'inférence ultérieure)
test_df_to_save = X_test_scaled.copy()
test_df_to_save['ACTUAL_PRICE'] = y_test.values

snowpark_test_df = session.create_dataframe(test_df_to_save)
snowpark_test_df.write.mode("overwrite").save_as_table("HOUSE_PRICE_TEST_DATA")

print("✅ Données de test sauvegardées dans HOUSE_PRICE_TEST_DATA")

---
## Étape 5 — Entraînement des modèles

Trois algorithmes sont comparés :
1. **Régression Linéaire** (baseline)
2. **Random Forest Regressor**
3. **XGBoost Regressor**

Métriques utilisées : **MAE**, **RMSE**, **R²**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def evaluate_regressor(model, X_train, X_test, y_train, y_test, model_name):
    """Entraîne un régresseur et retourne ses métriques."""
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        'model': model_name,
        'train_mae':  mean_absolute_error(y_train, y_pred_train),
        'test_mae':   mean_absolute_error(y_test, y_pred_test),
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'test_rmse':  np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'train_r2':   r2_score(y_train, y_pred_train),
        'test_r2':    r2_score(y_test, y_pred_test),
    }
    print(f"\n🔧 {model_name}")
    print(f"   Train R²: {metrics['train_r2']:.4f}  |  Test R²: {metrics['test_r2']:.4f}")
    print(f"   Train MAE: {metrics['train_mae']:,.0f}  |  Test MAE: {metrics['test_mae']:,.0f}")
    print(f"   Train RMSE: {metrics['train_rmse']:,.0f}  |  Test RMSE: {metrics['test_rmse']:,.0f}")
    return model, metrics

# --- Régression Linéaire ---
lr_model, lr_metrics = evaluate_regressor(
    LinearRegression(), X_train_scaled, X_test_scaled,
    y_train, y_test, "Régression Linéaire"
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# --- Random Forest ---
rf_model, rf_metrics = evaluate_regressor(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Random Forest Regressor"
)

In [ ]:
from xgboost import XGBRegressor

# --- XGBoost ---
xgb_model, xgb_metrics = evaluate_regressor(
    XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "XGBoost Regressor"
)

---
## Étape 6 — Évaluation comparative des modèles

In [ ]:
import streamlit as st
import altair as alt
import pandas as pd

all_metrics = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])

st.subheader("📊 Comparaison des modèles")
st.dataframe(
    all_metrics.style
        .highlight_max(subset=['test_r2'], color='#90EE90')
        .highlight_min(subset=['test_mae', 'test_rmse'], color='#90EE90')
        .format({
            'train_mae': '{:,.0f}', 'test_mae': '{:,.0f}',
            'train_rmse': '{:,.0f}', 'test_rmse': '{:,.0f}',
            'train_r2': '{:.4f}', 'test_r2': '{:.4f}'
        })
)

# Visualisation R²
r2_chart_data = all_metrics.melt(id_vars='model',
                                  value_vars=['train_r2', 'test_r2'],
                                  var_name='Split', value_name='R²')

chart_r2 = alt.Chart(r2_chart_data).mark_bar().encode(
    x=alt.X('model:N', title='Modèle'),
    y=alt.Y('R²:Q', scale=alt.Scale(domain=[0, 1])),
    color='Split:N',
    xOffset='Split:N',
    tooltip=['model', 'Split', alt.Tooltip('R²:Q', format='.4f')]
).properties(title='R² par modèle (Train vs Test)', width=500, height=300)

st.altair_chart(chart_r2, use_container_width=True)

# Résumé
best_initial = all_metrics.loc[all_metrics['test_r2'].idxmax()]
st.info(f"🏆 Meilleur modèle initial : **{best_initial['model']}** — Test R² = {best_initial['test_r2']:.4f}")

---
## Étape 7 — Optimisation des hyperparamètres (Grid Search)

On optimise **Random Forest** et **XGBoost** avec une grid search manuelle et le suivi `ExperimentTracking` Snowflake ML.

In [ ]:
from snowflake.ml.experiment.experiment_tracking import ExperimentTracking

exp = ExperimentTracking(session=session)
exp.set_experiment("House_Price_Experiment")
print("✅ Suivi d'expériences initialisé : House_Price_Experiment")

In [ ]:
import itertools
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Grille de paramètres Random Forest
rf_param_grid = {
    'n_estimators':    [100, 200],
    'max_depth':       [None, 10, 20],
    'min_samples_leaf':[1, 3],
    'max_features':    ['sqrt', 'log2']
}

rf_combinations = [
    dict(zip(rf_param_grid.keys(), v))
    for v in itertools.product(*rf_param_grid.values())
]

rf_results = []

for params in rf_combinations:
    params_full = {**params, 'random_state': 42, 'n_jobs': -1}
    run_name = f"RF_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    with exp.start_run(run_name):
        exp.log_params(params_full)
        model = RandomForestRegressor(**params_full)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)

        exp.log_metric("mae",  mae)
        exp.log_metric("rmse", rmse)
        exp.log_metric("r2",   r2)

        rf_results.append({
            'algorithm': 'RandomForest',
            **params,
            'mae': mae, 'rmse': rmse, 'r2': r2,
            '_model': model
        })
        print(f"  RF {params} → R²={r2:.4f}")

print("\n✅ Grid Search Random Forest terminé !")

In [ ]:
from xgboost import XGBRegressor

# Grille de paramètres XGBoost
xgb_param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0]
}

xgb_combinations = [
    dict(zip(xgb_param_grid.keys(), v))
    for v in itertools.product(*xgb_param_grid.values())
]

xgb_results = []

for params in xgb_combinations:
    params_full = {**params, 'random_state': 42, 'verbosity': 0}
    run_name = f"XGB_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    with exp.start_run(run_name):
        exp.log_params(params_full)
        model = XGBRegressor(**params_full)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2   = r2_score(y_test, y_pred)

        exp.log_metric("mae",  mae)
        exp.log_metric("rmse", rmse)
        exp.log_metric("r2",   r2)

        xgb_results.append({
            'algorithm': 'XGBoost',
            **params,
            'mae': mae, 'rmse': rmse, 'r2': r2,
            '_model': model
        })
        print(f"  XGB {params} → R²={r2:.4f}")

print("\n✅ Grid Search XGBoost terminé !")

In [ ]:
import streamlit as st
import pandas as pd

all_results = rf_results + xgb_results
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != '_model'} for r in all_results])

st.subheader("📊 Résultats de l'optimisation des hyperparamètres")
st.dataframe(
    results_df.sort_values('r2', ascending=False).head(10)
        .style.highlight_max(subset=['r2'], color='#90EE90')
        .highlight_min(subset=['mae', 'rmse'], color='#90EE90')
        .format({'mae': '{:,.0f}', 'rmse': '{:,.0f}', 'r2': '{:.4f}'})
)

# Sélection du meilleur modèle
best_idx = max(range(len(all_results)), key=lambda i: all_results[i]['r2'])
best_result = all_results[best_idx]
final_model = best_result['_model']

st.subheader("🏆 Meilleur modèle sélectionné")
st.write(f"**Algorithme :** {best_result['algorithm']}")
st.write(f"**R² (test) :** {best_result['r2']:.4f}")
st.write(f"**MAE (test) :** {best_result['mae']:,.0f} USD")
st.write(f"**RMSE (test) :** {best_result['rmse']:,.0f} USD")

params_display = {k: v for k, v in best_result.items()
                  if k not in ('algorithm', 'mae', 'rmse', 'r2', '_model')}
st.write("**Hyperparamètres :**", params_display)

---
## Étape 8 — Stockage du meilleur modèle dans le Model Registry

Enregistrement du modèle le plus performant dans le **Snowflake Model Registry** avec ses métriques et métadonnées.

In [ ]:
from snowflake.ml.registry import Registry
from datetime import datetime
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

registry = Registry(session)

MODEL_NAME    = "HOUSE_PRICE_PREDICTOR"
MODEL_VERSION = f"v_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# Nettoyage des noms de colonnes
X_train_clean = X_train_scaled.copy()
X_train_clean.columns = [c.upper() for c in X_train_clean.columns]

model_ref = registry.log_model(
    model=final_model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    sample_input_data=X_train_clean.head(5),
    metrics={
        'test_r2':   float(best_result['r2']),
        'test_mae':  float(best_result['mae']),
        'test_rmse': float(best_result['rmse'])
    },
    comment=f"Meilleur modèle ({best_result['algorithm']}) issu de l'optimisation des hyperparamètres. "
            f"R²={best_result['r2']:.4f}"
)

warnings.resetwarnings()

print(f"✅ Modèle enregistré dans le registry !")
print(f"   Nom    : {MODEL_NAME}")
print(f"   Version: {MODEL_VERSION}")

In [ ]:
-- Afficher tous les modèles du registry
SHOW MODELS;

In [ ]:
-- Afficher les versions du modèle
SHOW VERSIONS IN MODEL HOUSE_PRICE_PREDICTOR;

In [ ]:
# Affichage via l'API Python
registry.show_models()

---
## Étape 9 — Inférence : utiliser le modèle pour prédire

Chargement du modèle depuis le registry et application sur les données de test pour générer des prédictions.

In [ ]:
import streamlit as st
import altair as alt

# Chargement du modèle depuis le registry
loaded_model = registry.get_model(MODEL_NAME).version(MODEL_VERSION)

# Données de test
X_test_clean = X_test_scaled.copy()
X_test_clean.columns = [c.upper() for c in X_test_clean.columns]

# Prédictions via le modèle du registry
predictions_pd = loaded_model.run(X_test_clean, function_name="predict")

# Construction du tableau comparatif
results = X_test_clean.copy()
results['ACTUAL_PRICE']    = y_test.values
results['PREDICTED_PRICE'] = predictions_pd.values.flatten()
results['ERROR']           = results['ACTUAL_PRICE'] - results['PREDICTED_PRICE']
results['ABS_ERROR']       = results['ERROR'].abs()

st.subheader("📊 Prédictions vs Valeurs réelles")
st.dataframe(results[['ACTUAL_PRICE', 'PREDICTED_PRICE', 'ERROR', 'ABS_ERROR']]
             .head(20).style.format('{:,.0f}'))

# Scatter plot réel vs prédit
scatter_inf = alt.Chart(results.reset_index()).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('ACTUAL_PRICE:Q', title='Prix réel (USD)'),
    y=alt.Y('PREDICTED_PRICE:Q', title='Prix prédit (USD)'),
    tooltip=['ACTUAL_PRICE', 'PREDICTED_PRICE', 'ABS_ERROR']
).properties(title='Réel vs Prédit', width=500, height=350)

# Ligne parfaite
min_val = results[['ACTUAL_PRICE', 'PREDICTED_PRICE']].min().min()
max_val = results[['ACTUAL_PRICE', 'PREDICTED_PRICE']].max().max()
line_data = pd.DataFrame({'x': [min_val, max_val], 'y': [min_val, max_val]})
line_perfect = alt.Chart(line_data).mark_line(color='red', strokeDash=[5,5]).encode(
    x='x:Q', y='y:Q'
)

st.altair_chart(scatter_inf + line_perfect, use_container_width=True)

In [ ]:
# Sauvegarde des prédictions dans une table Snowflake
pred_df_to_save = results[['ACTUAL_PRICE', 'PREDICTED_PRICE', 'ERROR', 'ABS_ERROR']].copy()
pred_df_to_save = pred_df_to_save.reset_index(drop=True)  
session.create_dataframe(pred_df_to_save).write.mode("overwrite").save_as_table("HOUSE_PRICE_PREDICTIONS")

print("✅ Prédictions stockées dans HOUSE_PRICE_PREDICTIONS")

In [ ]:
-- Vérification des prédictions stockées
SELECT
    ROUND(ACTUAL_PRICE, 0)    AS PRIX_REEL,
    ROUND(PREDICTED_PRICE, 0) AS PRIX_PREDIT,
    ROUND(ABS_ERROR, 0)       AS ERREUR_ABSOLUE
FROM HOUSE_PRICE_PREDICTIONS
ORDER BY ABS_ERROR ASC
LIMIT 10;

---
## ✅ Conclusion

Ce notebook a couvert l'ensemble du pipeline **Data Engineering & ML** sur Snowflake :

| Étape | Réalisation |
|-------|-------------|
| 1. Configuration | Base de données, schéma, stage S3, file format |
| 2. Ingestion | Lecture CSV depuis S3, table `HOUSE_PRICE` |
| 3. EDA | Distribution, corrélation, visualisations Altair |
| 4. Préparation | Encodage, normalisation, split 80/20 |
| 5. Entraînement | Régression Linéaire, Random Forest, XGBoost |
| 6. Évaluation | MAE, RMSE, R² sur train/test |
| 7. Optimisation | Grid Search + ExperimentTracking Snowflake ML |
| 8. Registry | Modèle enregistré avec métriques et métadonnées |
| 9. Inférence | Prédictions stockées dans `HOUSE_PRICE_PREDICTIONS` |
| 10. Streamlit | Application interactive de prédiction |

> **Données utilisées :** `s3://logbrain-datalake/datasets/house_price/`  
> **Environnement :** Snowflake Notebook + Snowpark + Snowflake ML